# Norm64: four systems, exact times, and reference-based COMET

How often can a reader recover the exact time from both translations of a scenario? The four systems are compared on normalized inputs. The COMET comparison uses the two R4 input formats.

## Score the archived translations on CPU

**Run all** uses the Python standard library to total the recorded English readings and average the saved COMET scores. New translations and fresh COMET scoring are opt-in below.

Norm64 contains 64 messages / 32 A–B scenarios: **48 messages / 24 pairs** in Polish, Croatian and Russian, plus **16 messages / 8 pairs** in Ukrainian (no Ukrainian fine-tuning). Rules were developed on this same panel; it is not an independent holdout. “Outcome blind” describes original authorship, not untouched evaluation. The four normalized systems share the published normalized inputs. R4 adapter weights are not public: its translations, readings and COMET scores can be replayed, but R4 inference cannot be reproduced from this checkout.


In [1]:
from pathlib import Path
import json, hashlib, statistics, subprocess, tempfile

REPO_URL = "https://github.com/Silveroboros-dev/slavic-english-slm.git"
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "results/norm64_run.json").exists()), None)
if ROOT is None:
    # Colab fallback: only the public repository, never private artifacts or mounts.
    ROOT = Path(tempfile.mkdtemp(prefix="norm64-public-")) / "repo"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
manifest = json.loads((ROOT / "results/norm64_run.json").read_text())
raw = (ROOT / "results/norm64_cases.jsonl").read_bytes()
assert hashlib.sha256(raw).hexdigest() == manifest["case_file_sha256"]
rows = [json.loads(line) for line in raw.splitlines()]
assert len(rows) == len({r["id"] for r in rows}) == 64
for r in rows:
    assert (
        hashlib.sha256(r["normalized_input"].encode()).hexdigest() == r["normalized_input_sha256"]
    )
    for o in r["outputs"].values():
        assert hashlib.sha256(o["text"].encode()).hexdigest() == o["text_sha256"]
groups = {
    "PL/HR/RU": [r for r in rows if r["lang"] != "uk"],
    "UK": [r for r in rows if r["lang"] == "uk"],
}
assert [len(v) for v in groups.values()] == [48, 16]
print("Verified 64 public records and all archived text hashes; no model weights loaded.")

Verified 64 public records and all archived text hashes; no model weights loaded.


## Replay the frozen English readings

A translation passes when its archived English-only reading unambiguously recovers the expected 24-hour time. A scenario passes only if **both** A and B pass. A label with `status=exact` can still name the wrong time; the equality check below is essential. Unqualified “5:30” is ambiguous under this protocol. For Gemma’s “around 9:30 PM” in PL-07-B, the blinded reading was ambiguous, although the numeric checker accepted the clock value. The table counts blinded readings. It measures exact scheduling information on this panel.


In [2]:
SYSTEMS = ["nllb_base", "opus", "gemma_1b", "r4_normalized"]


def correct(row, system):
    label = row["outputs"][system]["label"]
    return label["status"] == "exact" and label["times_24h"] == [row["expected_24h"]]


def counts(records, system):
    pairs = {}
    for r in records:
        pairs.setdefault(r["pair_id"], []).append(r)
    assert all(
        len(pair) == 2 and {r["variant"] for r in pair} == {"A", "B"} for pair in pairs.values()
    )
    return sum(correct(r, system) for r in records), sum(
        all(correct(r, system) for r in pair) for pair in pairs.values()
    )


print("ARCHIVED READING REPLAY (messages; complete A/B pairs)")
for name, records in groups.items():
    print(name, f"n={len(records)}, pairs={len(records) // 2}")
    for system in SYSTEMS:
        a, b = counts(records, system)
        print(f"  {system:16s} {a}/{len(records)}; {b}/{len(records) // 2}")
assert [counts(groups["PL/HR/RU"], s)[1] for s in SYSTEMS] == [8, 1, 18, 22]

ARCHIVED READING REPLAY (messages; complete A/B pairs)
PL/HR/RU n=48, pairs=24
  nllb_base        21/48; 8/24
  opus             5/48; 1/24
  gemma_1b         39/48; 18/24
  r4_normalized    46/48; 22/24
UK n=16, pairs=8
  nllb_base        10/16; 4/8
  opus             3/16; 0/8
  gemma_1b         15/16; 7/8
  r4_normalized    16/16; 8/8


In [3]:
print("Example: exact clock recovery does not imply faithful nouns")
r = rows[0]
print("Source:", r["normalized_input"])
print("Reference:", r["reference"])
for s in SYSTEMS:
    print(s, ":", r["outputs"][s]["text"])
    print("  archived reading:", r["outputs"][s]["label"], "pass:", correct(r, s))

Example: exact clock recovery does not imply faithful nouns
Source: Cysterna z mlekiem podjeżdża pod mleczarnię o 05:30. Przygotuj wtedy wąż do rozładunku.
Reference: The milk tanker arrives at the dairy at 05:30. Have the unloading hose ready then.
nllb_base : The milk tank will be dropped off at the dairy at 5:30, then prepare the snake for unloading.
  archived reading: {'status': 'ambiguous', 'times_24h': [], 'clock12': ['05:30'], 'evidence': '5:30', 'output_class': 'english'} pass: False
opus : The milk truck comes into the milk shop at 05:30.
  archived reading: {'status': 'exact', 'times_24h': ['05:30'], 'clock12': ['05:30'], 'evidence': '05:30', 'output_class': 'english'} pass: True
gemma_1b : A bovine crate approaches the dairy shop at 5:30 AM. Then, prepare a winch for unloading.
  archived reading: {'status': 'exact', 'times_24h': ['05:30'], 'clock12': ['05:30'], 'evidence': '5:30 AM', 'output_class': 'english'} pass: True
r4_normalized : The milk carton drives under the dai

## Replay COMET on the R4 spoken versus normalized arms

These are archived reference-based `Unbabel/wmt22-comet-da` scores, not a fresh scorer run. Both arms used the **same original spoken source and English reference**; only the translated English differed. Averaging here does not call COMET or establish exact-time correctness. The original CPU scorer used revision `2760a223ac957f30acfb18c8aa649b01cf1d75f2`, float32, seed 1234, two CPU threads, batches of eight distinct triples, and XLM-R large revision `c23d21b0620b635a76227c604d44e43a9f0ee389`.


In [4]:
for name, records in groups.items():
    before = statistics.mean(r["outputs"]["r4_spoken"]["comet"] for r in records)
    after = statistics.mean(r["outputs"]["r4_normalized"]["comet"] for r in records)
    print(
        f"{name} n={len(records)}: archived COMET {before:.5f} -> {after:.5f}; delta {after - before:+.5f}"
    )
    print(
        "  R4 complete clock pairs:",
        counts(records, "r4_spoken")[1],
        "->",
        counts(records, "r4_normalized")[1],
    )

PL/HR/RU n=48: archived COMET 0.79107 -> 0.78974; delta -0.00134
  R4 complete clock pairs: 0 -> 22
UK n=16: archived COMET 0.79223 -> 0.80253; delta +0.01030
  R4 complete clock pairs: 0 -> 8


## Optional inference with base NLLB and OPUS-MT

Disabled by default. In a **separate inference environment**, install `torch>=2.6`, `transformers==4.57.1`, `tokenizers==0.22.2`, `sentencepiece`, `sacremoses` and `huggingface_hub`. Set `RUN_FRESH_MT=True` below. This downloads the original public NLLB and OPUS checkpoints and generates on the public normalized messages. CPU float32 is the default and can be slow; change `DEVICE` deliberately for GPU. Revisions, source language tags, English direction, greedy decoding and batch size match the original recipe; CPU versus archived NVIDIA hardware may change outputs. OPUS is the multilingual **sla-en** model; the audited original passed raw normalized source strings without an added OPUS language prefix. NLLB explicitly uses its four source tags and forces `eng_Latn`.

The default subset contains one complete pair per language. Set `FRESH_LIMIT=None` for all 64. New predictions are saved separately with no inherited archived reading labels or scores. New English-only readings are needed before claiming new exact-time counts. Gemma is deliberately omitted: the archive used a Kaggle version and frozen chat-template hash, so substituting a generic Hugging Face identifier would not establish the same experiment. No R4 checkpoint is downloaded or inferred.


In [5]:
RUN_FRESH_MT = False
DEVICE = "cpu"
FRESH_LIMIT = 8  # None means the full public panel
fresh_predictions = []
if RUN_FRESH_MT:
    import torch, transformers
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, MarianTokenizer, MarianMTModel
    from huggingface_hub import snapshot_download
    from importlib.metadata import version

    assert transformers.__version__ == "4.57.1"
    assert version("tokenizers") == "0.22.2"
    torch.manual_seed(42)
    torch.set_num_threads(2)
    chosen = (
        [r for r in rows if r["pair_id"].endswith("-01")]
        if FRESH_LIMIT == 8
        else rows[:FRESH_LIMIT]
    )
    tags = {"pl": "pol_Latn", "hr": "hrv_Latn", "ru": "rus_Cyrl", "uk": "ukr_Cyrl"}
    for system in ["nllb_base", "opus"]:
        spec = manifest["systems"][system]
        nllb = system == "nllb_base"
        folder = Path(
            snapshot_download(
                spec["model"],
                revision=spec["revision"],
                allow_patterns=["*.json", "*.spm", "*.model", "pytorch_model.bin"],
            )
        )
        weight = folder / "pytorch_model.bin"
        with weight.open("rb") as f:
            assert hashlib.file_digest(f, "sha256").hexdigest() == spec["weights_sha256"]
        tok = (
            AutoTokenizer.from_pretrained(
                folder,
                use_fast=True,
                legacy_behaviour=False,
                tgt_lang="eng_Latn",
                local_files_only=True,
            )
            if nllb
            else MarianTokenizer.from_pretrained(folder, local_files_only=True)
        )
        cls = AutoModelForSeq2SeqLM if nllb else MarianMTModel
        model = (
            cls.from_pretrained(
                folder, use_safetensors=False, dtype=torch.float32, local_files_only=True
            )
            .to(DEVICE)
            .eval()
        )
        with torch.inference_mode():
            for lang, tag in tags.items():
                subset = [r for r in chosen if r["lang"] == lang]
                if nllb:
                    tok.src_lang, tok.tgt_lang = tag, "eng_Latn"
                for i in range(0, len(subset), 2):
                    batch = subset[i : i + 2]
                    enc = tok(
                        [r["normalized_input"] for r in batch],
                        padding=True,
                        truncation=False,
                        return_tensors="pt",
                    )
                    assert enc.input_ids.shape[1] <= (
                        512 if nllb else model.config.max_position_embeddings
                    )
                    kwargs = {}
                    if nllb:
                        assert bool((enc.input_ids[:, 0] == tok.convert_tokens_to_ids(tag)).all())
                        kwargs["forced_bos_token_id"] = tok.convert_tokens_to_ids("eng_Latn")
                    seq = model.generate(
                        **enc.to(DEVICE),
                        num_beams=1,
                        do_sample=False,
                        num_return_sequences=1,
                        max_new_tokens=256,
                        use_cache=True,
                        **kwargs,
                    )
                    for r, text in zip(batch, tok.batch_decode(seq, skip_special_tokens=True)):
                        fresh_predictions.append(
                            {
                                "id": r["id"],
                                "system": system,
                                "normalized_input": r["normalized_input"],
                                "text": text,
                                "kind": "fresh_generation",
                                "model": spec["model"],
                                "revision": spec["revision"],
                                "device": DEVICE,
                                "dtype": "float32",
                                "num_beams": 1,
                                "do_sample": False,
                                "max_new_tokens": 256,
                                "transformers": transformers.__version__,
                            }
                        )
        del model
    fresh_path = ROOT / "fresh_norm64_baseline_predictions.json"
    fresh_path.write_text(json.dumps(fresh_predictions, ensure_ascii=False, indent=2))
    print("NEW inference results:", len(fresh_predictions), fresh_path.name)
else:
    print("Fresh inference disabled: no imports, model downloads or new translations.")

Fresh inference disabled: no imports, model downloads or new translations.


## Optional COMET scoring of the saved R4 translations

This is new **scorer inference on archived translations**, not new R4 translation inference. It uses public rows only. Run in a separate Python 3.11 environment with `torch==2.6.0` (CPU wheel), `unbabel-comet==2.2.7`, `transformers==4.51.3`, `pytorch-lightning==2.2.5`, `huggingface_hub` and `sentencepiece`; the COMET and translation recipes intentionally have different Transformers versions. Restart the kernel and replay the stdlib cells after changing environments. Set the flag below to download the large scorer. The loader follows the original CPU notebook: pinned checkpoint and encoder files, no replacement encoder weights, float32, two threads, seed 1234, batches of eight deduplicated triples. It checks checkpoint hash and tolerates only nonparameter position/token-type ID buffers. The checkpoint must be trusted because its original format needs `weights_only=False`.

The default scores four complete pairs (16 arm outputs); set `COMET_ALL=True` for all 128. Both arms always use `spoken_source`, never the normalized source in one arm. Results are written separately from the frozen archive. Exact numerical equality can depend on runtime versions and hardware; this notebook does not silently replace archived scores.


In [6]:
RUN_FRESH_COMET = False
COMET_ALL = False
if RUN_FRESH_COMET:
    import torch, inspect, math
    from importlib.metadata import version
    from huggingface_hub import hf_hub_download
    from comet.models.regression.regression_metric import RegressionMetric

    for package, expected in {
        "unbabel-comet": "2.2.7",
        "transformers": "4.51.3",
        "pytorch-lightning": "2.2.5",
    }.items():
        assert version(package) == expected, (package, version(package))
    assert torch.__version__.split("+")[0] == "2.6.0"
    spec = manifest["metrics"]["comet"]
    ckpt = Path(
        hf_hub_download(spec["model"], revision=spec["revision"], filename="checkpoints/model.ckpt")
    )
    with ckpt.open("rb") as f:
        assert hashlib.file_digest(f, "sha256").hexdigest() == spec["weights_sha256"]
    backbone_rev = "c23d21b0620b635a76227c604d44e43a9f0ee389"
    for filename in [
        "config.json",
        "tokenizer.json",
        "tokenizer_config.json",
        "sentencepiece.bpe.model",
    ]:
        encoder_file = hf_hub_download(
            "FacebookAI/xlm-roberta-large", revision=backbone_rev, filename=filename
        )
    encoder = str(Path(encoder_file).parent)
    try:
        checkpoint = torch.load(ckpt, map_location="cpu", weights_only=False, mmap=True)
    except RuntimeError as error:
        if "mmap can only be used" not in str(error):
            raise
        checkpoint = torch.load(ckpt, map_location="cpu", weights_only=False)
    params = dict(checkpoint["hyper_parameters"])
    params.update(pretrained_model=encoder, load_pretrained_weights=False, local_files_only=True)
    if params.get("layer_transformation") == "sparsemax_patch":
        params["layer_transformation"] = "softmax"
    params = {
        k: v
        for k, v in params.items()
        if k in inspect.signature(RegressionMetric.__init__).parameters
    }
    torch.manual_seed(1234)
    scorer = RegressionMetric(**params)
    missing, unexpected = scorer.load_state_dict(checkpoint["state_dict"], strict=False)
    assert not [
        k for k in missing + unexpected if not k.endswith((".position_ids", ".token_type_ids"))
    ]
    del checkpoint
    torch.manual_seed(1234)
    torch.set_num_threads(2)
    torch.set_float32_matmul_precision("highest")
    scorer = scorer.float().cpu().eval()
    chosen = rows if COMET_ALL else [r for r in rows if r["pair_id"].endswith("-01")]
    triples, identities = [], []
    for r in chosen:
        for arm in ["r4_spoken", "r4_normalized"]:
            triples.append((r["spoken_source"], r["outputs"][arm]["text"], r["reference"]))
            identities.append((r["id"], arm))
    for i in range(0, len(triples), 2):
        assert triples[i][0] == triples[i + 1][0] and triples[i][2] == triples[i + 1][2]
    unique = list(dict.fromkeys(triples))
    values = {}
    with torch.inference_mode():
        for i in range(0, len(unique), 8):
            batch = unique[i : i + 8]
            prepared = scorer.prepare_sample(
                [dict(zip(["src", "mt", "ref"], t)) for t in batch], stage="predict"
            )
            scores = scorer.predict_step(prepared, i // 8).scores.detach().float().tolist()
            assert len(scores) == len(batch) and all(math.isfinite(s) for s in scores)
            values.update(zip(batch, scores))
    result = [
        {"id": ident[0], "arm": ident[1], "score": values[t]}
        for ident, t in zip(identities, triples)
    ]
    payload = {
        "kind": "fresh_COMET_on_archived_public_R4_texts",
        "model": spec,
        "backbone_revision": backbone_rev,
        "source_policy": "identical original spoken source and reference",
        "seed": 1234,
        "batch_size": 8,
        "threads": 2,
        "dtype": "float32",
        "device": "cpu",
        "versions": {
            p: version(p) for p in ["torch", "transformers", "unbabel-comet", "pytorch-lightning"]
        },
        "case_file_sha256": manifest["case_file_sha256"],
        "rows": result,
    }
    out = ROOT / "fresh_norm64_r4_comet.json"
    out.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
    for arm in ["r4_spoken", "r4_normalized"]:
        print("FRESH COMET", arm, statistics.mean(r["score"] for r in result if r["arm"] == arm))
    print("New scorer results:", out.name)
else:
    print("Fresh COMET disabled: archived scores above were replayed, not recomputed.")

Fresh COMET disabled: archived scores above were replayed, not recomputed.
